# Part 3: Add Work IQ as a knowledge source

Work IQ is a **first-party knowledge source kind** (`workIQ`) on the
`2026-05-01-preview` API. This notebook creates a Work IQ KS, wires it into a
KB, and runs a grounded retrieve.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory (same folder as this notebook) — `BIGBUGS_ENDPOINT`, `BIGBUGS_API_KEY`. |
| Work IQ token | A valid `workiq token` passed as `x-ms-query-source-authorization: accessToken-<token>`. |

> ⚠️ **The `.env` is currently in testing state — ask Matt for setup credentials.**
> Never check secrets into this repo.

### Current status on BigBugs

- WorkIQ KS CRUD ✅
- KB create + retrieve ✅ (but **intermittent** — may hit 30s timeout or downstream
  `ServiceUnavailable`). When it works, you get a full `200 OK` with `response`,
  `activity`, and `references`.
- Auth: requires a real Work IQ token. The notebook will try `workiq token` CLI
  if available, otherwise you must set `WORKIQ_QUERY_SOURCE_AUTHORIZATION` in .env.

## 1 — Setup

In [ ]:
import json, os, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone

import requests

ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(r: requests.Response):
    print(f"{r.status_code} {r.reason}")
    try:
        print(json.dumps(r.json(), indent=2))
    except Exception:
        print(r.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
KS_NAME = f"lab532-workiq-{stamp}"
KB_NAME = f"lab532-workiq-kb-{stamp}"

print(f"Endpoint : {ENDPOINT}")
print(f"KS       : {KS_NAME}")
print(f"KB       : {KB_NAME}")

## 2 — Obtain a Work IQ token

The retrieve call needs `x-ms-query-source-authorization: accessToken-<token>`.
We try the `workiq token` CLI first, then fall back to the .env variable.

In [ ]:
def get_workiq_token(env: dict) -> str:
    # Try env var first
    raw = env.get("WORKIQ_QUERY_SOURCE_AUTHORIZATION") or env.get("X_MS_QUERY_SOURCE_AUTHORIZATION")
    if raw:
        token = raw.strip()
        if not token.startswith("accessToken-"):
            token = f"accessToken-{token}"
        print("Using Work IQ token from .env")
        return token

    # Try CLI
    try:
        result = subprocess.run(["workiq", "token"], check=True,
                                capture_output=True, text=True)
        token = result.stdout.strip()
        if token:
            if not token.startswith("accessToken-"):
                token = f"accessToken-{token}"
            print("Using Work IQ token from `workiq token` CLI")
            return token
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass

    raise RuntimeError(
        "No Work IQ token found. Set WORKIQ_QUERY_SOURCE_AUTHORIZATION in .env "
        "or ensure the `workiq` CLI is available."
    )

WORKIQ_TOKEN = get_workiq_token(env)
print(f"Token length: {len(WORKIQ_TOKEN)} chars")

## 3 — Create the `workIQ` knowledge source

In [ ]:
ks_body = {
    "name": KS_NAME,
    "kind": "workIQ",
    "description": "Lab 532 Work IQ source",
}

r = session.put(url(f"/knowledgesources('{KS_NAME}')"), json=ks_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 4 — Create a KB

In [ ]:
kb_body = {
    "name": KB_NAME,
    "description": "Lab 532 KB — Work IQ backed",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
    "knowledgeSources": [{"name": KS_NAME}],
}

r = session.put(url(f"/knowledgebases('{KB_NAME}')"), json=kb_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 5 — Retrieve: ask a Work IQ question

> ⚠️ This call may **time out** or return `ServiceUnavailable` — the downstream
> Work IQ service is intermittent on this stamp. If it fails, try re-running.

In [ ]:
retrieve_body = {
    "includeActivity": True,
    "intents": [
        {"type": "semantic",
         "search": "Summarize recent work on Azure AI Search."}
    ],
    "knowledgeSourceParams": [
        {
            "kind": "workIQ",
            "knowledgeSourceName": KS_NAME,
            "includeReferenceSourceData": True,
            "includeReferences": True,
        }
    ],
    "maxRuntimeInSeconds": 90,
}

r = session.post(
    url(f"/knowledgebases('{KB_NAME}')/retrieve"),
    json=retrieve_body,
    headers={"x-ms-query-source-authorization": WORKIQ_TOKEN},
    timeout=120,
)
pp(r)

if r.status_code == 200:
    print("\n✅ Work IQ retrieve succeeded!")
else:
    print("\n⚠️ Work IQ retrieve failed — this is a known intermittent issue.")
    print("   Try re-running or check downstream Work IQ service health.")

## 6 — Cleanup

In [ ]:
for label, path in [
    ("KB", f"/knowledgebases('{KB_NAME}')"),
    ("KS", f"/knowledgesources('{KS_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Known issues

| Issue | Detail |
|-------|--------|
| Intermittent timeouts | Downstream Work IQ service hits 30s timeout or `ServiceUnavailable`. |
| Token source | The `workiq token` CLI may not be available — set the env var as fallback. |
| No MCP/A2A fallback needed | Work IQ is a first-party `workIQ` kind — no adapter required. |

➡️ Continue to [Part 4: Add a web source to the KB](part4-web-source-to-kb.ipynb).